# 模型归因分析 v1

目标：验证 `src.analysis.explain_model_artifact` 的稳定归因接口。覆盖 native linear、native boosting、proxy graph 三类模型，输出 feature attribution sidecar 和中文结论。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "SPEC.md").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("cannot locate repo root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD = ROOT / "research" / "model_attribution_analysis_v1"
CARD.mkdir(parents=True, exist_ok=True)

START = "2026-01-01"
END = "2026-06-09"
TRAIN_END = "2026-04-30"
MODEL_NAMES = ("ridge_mse", "xgboost_mse", "gcn_mse")
CARD

PosixPath('/Users/rziiiii/dev/workspace/research/model_attribution_analysis_v1')

In [2]:
from src.analysis import explain_model_artifact
from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model import fit_model_artifact
from src.model.capabilities import model_capability_table

price_frame = load_soxl_support_frame(start=START, end=END)
dataset = build_supervised_dataset(price_frame, horizon=1)
train = dataset.frame.loc[dataset.frame["date"] <= TRAIN_END].copy()
test = dataset.frame.loc[dataset.frame["date"] > TRAIN_END].copy()
feature_columns = dataset.feature_columns[:8]

assert not train.empty and not test.empty
train.shape, test.shape, feature_columns

((82, 18),
 (26, 18),
 ('soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d',
  'vix_level',
  'vix_change_1d',
  'intraday_range'))

In [3]:
reports = []
summaries = []
for model_name in MODEL_NAMES:
    artifact = fit_model_artifact(model_name, train=train, feature_columns=feature_columns)
    report = explain_model_artifact(artifact, frame=test, background_frame=train, top_n=6)
    reports.append(report.feature_attribution)
    summaries.append(report.summary)

feature_attribution = pd.concat(reports, ignore_index=True)
feature_attribution_path = CARD / "feature_attribution.csv"
feature_attribution.to_csv(feature_attribution_path, index=False)

model_summaries_path = CARD / "model_attribution_summaries.json"
model_summaries_path.write_text(json.dumps(summaries, ensure_ascii=False, indent=2), encoding="utf-8")

assert set(feature_attribution["model"]) == set(MODEL_NAMES)
assert {"permutation", "sensitivity"}.issubset(set(feature_attribution["attribution_kind"]))
assert "coefficient" in set(feature_attribution.loc[feature_attribution["model"].eq("ridge_mse"), "attribution_kind"])
feature_attribution.head(12)

,model,feature,attribution_kind,importance,normalized_importance,rank,direction
0,ridge_mse,soxx_return_5d,coefficient,0.032091,0.310861,1,-1.0
1,ridge_mse,soxl_soxx_relative_5d,coefficient,0.030108,0.291646,2,1.0
2,ridge_mse,soxx_return_1d,coefficient,0.013650,0.132224,3,1.0
3,ridge_mse,vix_change_1d,coefficient,0.010937,0.105945,4,1.0
4,ridge_mse,soxl_return_5d,coefficient,0.010432,0.101053,5,1.0
5,ridge_mse,vix_level,coefficient,0.006015,0.058271,6,1.0
6,ridge_mse,soxl_soxx_relative_5d,permutation,0.043800,0.296345,1,1.0
7,ridge_mse,soxx_return_5d,permutation,0.041476,0.280621,2,1.0
8,ridge_mse,soxx_return_1d,permutation,0.027973,0.189260,3,-1.0
9,ridge_mse,soxl_return_5d,permutation,0.014461,0.097840,4,-1.0


In [4]:
capabilities = model_capability_table(MODEL_NAMES)
capabilities_path = CARD / "model_capabilities.csv"
capabilities.to_csv(capabilities_path, index=False)

top_by_model = (
    feature_attribution.groupby(["model", "feature"], as_index=False)["normalized_importance"].sum()
    .sort_values(["model", "normalized_importance"], ascending=[True, False])
    .groupby("model")
    .head(3)
)
top_features_path = CARD / "top_features_by_model.csv"
top_by_model.to_csv(top_features_path, index=False)

analysis_summary = {
    "status": "completed",
    "schema_version": "analysis.model_attribution_notebook.v1",
    "models": list(MODEL_NAMES),
    "train_end": TRAIN_END,
    "test_start": str(test["date"].min().date()),
    "test_end": str(test["date"].max().date()),
    "feature_count": int(len(feature_columns)),
    "attribution_kinds": sorted(feature_attribution["attribution_kind"].unique().tolist()),
    "proxy_models": capabilities.loc[capabilities["is_proxy"].astype(bool), "model"].tolist(),
    "artifacts": {
        "feature_attribution": str(feature_attribution_path.relative_to(CARD)),
        "model_attribution_summaries": str(model_summaries_path.relative_to(CARD)),
        "model_capabilities": str(capabilities_path.relative_to(CARD)),
        "top_features_by_model": str(top_features_path.relative_to(CARD)),
    },
    "risk_notes": [
        "Permutation 和 sensitivity 是局部解释工具，不等价于因果解释。",
        "gcn_mse 当前仍为 proxy_feature_adapter，归因解释的是 proxy feature path。",
        "本次只覆盖 2026-05-01 至 2026-06-08 测试窗，需要在更长 walk-forward 中复验。",
    ],
}
(CARD / "analysis_summary.json").write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2), encoding="utf-8")
analysis_summary

{'status': 'completed',
 'schema_version': 'analysis.model_attribution_notebook.v1',
 'models': ['ridge_mse', 'xgboost_mse', 'gcn_mse'],
 'train_end': '2026-04-30',
 'test_start': '2026-05-01',
 'test_end': '2026-06-08',
 'feature_count': 8,
 'attribution_kinds': ['coefficient',
  'feature_importance',
  'permutation',
  'sensitivity'],
 'proxy_models': ['gcn_mse'],
 'artifacts': {'feature_attribution': 'feature_attribution.csv',
  'model_attribution_summaries': 'model_attribution_summaries.json',
  'model_capabilities': 'model_capabilities.csv',
  'top_features_by_model': 'top_features_by_model.csv'},
 'risk_notes': ['Permutation 和 sensitivity 是局部解释工具，不等价于因果解释。',
  'gcn_mse 当前仍为 proxy_feature_adapter，归因解释的是 proxy feature path。',
  '本次只覆盖 2026-05-01 至 2026-06-08 测试窗，需要在更长 walk-forward 中复验。']}

## 结论

- 本次在 `ridge_mse`、`xgboost_mse`、`gcn_mse` 上验证了统一模型归因接口；输出包含 `feature_attribution.csv`、`top_features_by_model.csv`、`model_capabilities.csv` 和 `analysis_summary.json`。
- `ridge_mse` 同时具备 coefficient、permutation、sensitivity 三类归因；`xgboost_mse` 和 `gcn_mse` 至少具备 permutation 与 sensitivity，满足 tabular/native 与 proxy graph 的共同解释入口。
- `gcn_mse` 当前是 `proxy_feature_adapter`，归因结果只能解释现有 proxy feature path，不能解释完整论文 GCN 结构。
- 不晋升 production：本次只验证分析接口和一个短测试窗。下一步应把 attribution sidecar 接入 dashboard manifest，并在 walk-forward/regime 分段中比较 top feature 稳定性。